# Mounting Drive to get the PPT data access
### Make sure to create a shortcut of the shared "GenAI (UI UX)" folder, directly in your drive (not in any other folder),   
###to seamlessly run the pipeline           
###Otherwise refer the address block below ###


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Installing required libraries


###*   Sentence Transformers - For query embeddings
###*   ChromaDB - To access vector database
###*   FAISS-CPU - To retrieve slides based on AI similarity search
###*   PyMuPDF - To access and manipulate PDF files
###*   ConvertAPI - To convert PDF to PPT at the end
###*   Imagehash - To remove unnecessary slides from getting selected





In [ ]:
!pip install sentence-transformers chromadb faiss-cpu PyMuPDF convertapi imagehash -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 5.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.3/584.3 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.5/296.5 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 6.4 MB/s eta

# Address block
###Update the address based on your setup.

###Initially, the addresses are according to the common shared folder [ GenAI (UI/UX) ]                                                                     
###So if you are using the shared folder, run this cell and proceed.###


In [ ]:
database_directory = "/content/drive/MyDrive/GenAI (UI UX)/PPT_DB"

json_files_path = "/content/drive/MyDrive/GenAI (UI UX)/Json_Files/Final_Files"

ppt_data_path = "/content/drive/MyDrive/GenAI (UI UX)/PPT_DATA"

output_pdf_path = "/content/drive/MyDrive/GenAI (UI UX)/OUTPUT_FILES/OUTPUT_PDF"

output_ppt_path = "/content/drive/MyDrive/GenAI (UI UX)/OUTPUT_FILES/OUTPUT_PPT"

# Initialising ChromaDB (new or existing), and getting/creating the collection.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
from chromadb.utils import embedding_functions

sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="intfloat/e5-large-v2")

new_settings = Settings(
    chroma_api_impl="chromadb.api.segment.SegmentAPI",
    is_persistent=True,
    persist_directory=database_directory,
    chroma_server_ssl_enabled=True,
    chroma_server_host="localhost",
    chroma_server_http_port=8080,
    anonymized_telemetry=False
)

chroma_client = chromadb.PersistentClient(path = database_directory, settings = new_settings)

collection = chroma_client.get_or_create_collection("Slides", embedding_function = sentence_transformer_ef)

hf_model = SentenceTransformer('intfloat/e5-large-v2')



# Use only to create ChromaDB from scratch using JSON files


In [ ]:
import json
from typing import List, Dict
import os

# Load and process the JSON data
def load_and_process_json(file_path: str) -> List[Dict]:
    with open(file_path, 'r') as file:
        data = json.load(file)

    processed_data = []
    for item in data['output']:
        filename = item['filename']
        for slide in item['content']:
            processed_data.append({
                'id': f"{filename}_{slide['slide_number']}",
                'text': slide['text'],
                'images_summary' : slide['images_summary'],
                'metadata': {
                    'filename': filename,
                    'slide_number': slide['slide_number']
                }
            })
    return processed_data

# Store data in Chroma
def store_in_chroma(data: List[Dict]):
    ids = [item['id'] for item in data]
    texts = [f"Text:\n{item['text']}\n\nImages Summary:\n{item['images_summary']}" for item in data]
    metadatas = [item['metadata'] for item in data]

    collection.add(
        ids=ids,
        documents=texts,
        metadatas=metadatas
    )

# Main execution (From Scratch)
for files in os.listdir(json_files_path):
  json_file_path = os.path.join(json_files_path, files)
  processed_data = load_and_process_json(json_file_path)
  store_in_chroma(processed_data)

# Use this to add any slides to the ChromaDB


In [ ]:
json_file_path = ""  # path to your JSON file
processed_data = load_and_process_json(json_file_path)
store_in_chroma(processed_data)

# To seperate slide content into text, table and image summary, for further scoring

In [ ]:
import re

def separate_content(text):
    # Remove any surrounding quotes
    text = text.strip("'\"")

    # Split the content into sections
    sections = re.split(r'\n(?=Text:|\*\*Table\*\*:|Images Summary:)', text)
    text_content = ""
    table_content = ""
    images_summary_content = ""

    for section in sections:
        if section.startswith("Text:"):
            text_content = section.replace("Text:", "").strip()
        elif section.startswith("**Table**:"):
            table_content = section.replace("**Table**:", "").strip()
        elif section.startswith("Images Summary:"):
            images_summary_content = section.replace("Images Summary:", "").strip()

    # Clean up the images summary content
    images_summary_content = images_summary_content.strip("[]")

    return text_content, table_content, images_summary_content


# To get the most different slide combinations

In [ ]:
from itertools import combinations
import numpy as np

def calculate_distance(slide1, slide2):
    return abs(slide1['combined_score'] - slide2['combined_score'])

# Function to find the slides that are farthest from each other
def find_farthest_slides(slides, n):
    max_distance = -1
    best_combination = None

    for combo in combinations(slides, n):
        total_distance = 0

        # Calculate the pairwise distances for the combination
        for i in range(len(combo)):
            for j in range(i + 1, len(combo)):
                total_distance += calculate_distance(combo[i], combo[j])

        if total_distance > max_distance:
            max_distance = total_distance
            best_combination = combo

    return best_combination

# Main Retrieval Function

$$\text{Relevance Score} = 0.6 \times (\text{Text Relevance Score}) + 0.2 \times (\text{Table Relevance Score}) + 0.2 \times (\text{Image Relevance Score})$$

$$\text{Similarity Score} = \left(\frac{1}{1 + \text{Distance}} - 0.5\right) \times 2$$

$$\text{Combined Score} = \alpha \times (\text{Relevance Score}) + (1 - \alpha) \times (\text{Similarity Score})$$


In [ ]:
import imagehash
import io
import faiss
from PIL import Image
import numpy as np
import torch
import os
from typing import List, Dict
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch.nn.functional as F

# Initialising the model that calculates relevance score
model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
relevance_model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Function to calculate relevance score
def get_relevance_score(query, document):
    inputs = tokenizer(query, document, return_tensors="pt", truncation=True, max_length=512, padding=True)
    with torch.no_grad():
        outputs = relevance_model(**inputs)

    logits = outputs.logits.squeeze()

    if logits.ndim == 0:
        logits = logits.unsqueeze(0)

    if logits.shape[0] == 2:
        return F.softmax(logits, dim=0)[1].item()
    elif logits.shape[0] == 1:
        return torch.sigmoid(logits).item()
    else:
        return F.softmax(logits, dim=0).max().item()

# Query Chroma function
def query_chroma(queries: List[Dict], use_hnsw: bool = True, normalize: bool = True, filenames: List[str] = None):
    all_results = []
    best_result = []
    excluded_files = set()
    used_hashes = set()
    unique_items = []

# Function to filter unique images using image perceptual hashing
    def filter_unique_images(image_paths, used_hashes, threshold=5):

        unique_images = {}

        for image_path,value in image_paths.items():

            image = Image.open(image_path)
            # Calculate perceptual hash (phash) for the image
            phash = imagehash.phash(image)

            # Check against all used hashes
            is_unique = True
            for used_hash in used_hashes:
                if phash - used_hash <= threshold:
                    is_unique = False
                    break

            # If the image is unique, add it to the unique_images dict
            if is_unique:
                unique_images[image_path] = value
                used_hashes.add(phash)  # Optionally add this hash to used_hashes

        return unique_images

    for query in queries:
      for query_text, num_of_slides in query.items():
        # Use if you want to retrieve slides from some specific PPTs
        if filenames:
            all_data = collection.get(include=['embeddings', 'documents', 'metadatas'])
            filtered_data = {
                'embeddings': [],
                'documents': [],
                'metadatas': [],
                'ids': []
            }
            for embedding, document, metadata,ids in zip(all_data['embeddings'], all_data['documents'], all_data['metadatas'],all_data['ids']):
                text_content, table_content, images_summary_content = separate_content(document)
                if len(text_content) > 150:
                    filtered_data['embeddings'].append(embedding)
                    filtered_data['documents'].append(document)
                    filtered_data['metadatas'].append(metadata)
                    filtered_data['ids'].append(ids)
            all_data = filtered_data

            for i, metadata in enumerate(all_data['metadatas']):
                if metadata['filename'] in filenames:
                    filtered_data['embeddings'].append(all_data['embeddings'][i])
                    filtered_data['documents'].append(all_data['documents'][i])
                    filtered_data['metadatas'].append(all_data['metadatas'][i])
                    filtered_data['ids'].append(all_data['ids'][i])
            all_data = filtered_data
        # Normally, executes from here
        else:
            all_data = collection.get(include=['embeddings', 'documents', 'metadatas'])
            filtered_data = {
                'embeddings': [],
                'documents': [],
                'metadatas': [],
                'ids': []
            }

            for embedding, document, metadata,ids in zip(all_data['embeddings'], all_data['documents'], all_data['metadatas'],all_data['ids']):
                text_content, table_content, images_summary_content = separate_content(document)
                if len(text_content) > 150:
                    filtered_data['embeddings'].append(embedding)
                    filtered_data['documents'].append(document)
                    filtered_data['metadatas'].append(metadata)
                    filtered_data['ids'].append(ids)
            all_data = filtered_data

        embeddings = np.array(all_data['embeddings']).astype('float32')

        # To normalize the embeddings
        if normalize:
            if embeddings.ndim == 1:
                embeddings = embeddings / np.linalg.norm(embeddings)
            else:
                embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

        if embeddings.ndim == 1:
            embeddings = embeddings.reshape(1, -1)

        dimension = embeddings.shape[1]

        # Initialize the index
        if use_hnsw:
            M = 16
            index = faiss.IndexHNSWFlat(dimension, M)
            index.hnsw.efConstruction = 40
            index.hnsw.efSearch = 32
        else:
            index = faiss.IndexFlatL2(dimension)

        index.add(embeddings)

        # Getting embeddings for the query
        query_embedding = hf_model.encode(query_text)
        if normalize:
            query_embedding = query_embedding / np.linalg.norm(query_embedding)

        query_embedding = query_embedding.reshape(1, -1)

        if query_embedding.shape[1] != dimension:
            print(f"Error: Query embedding dimension ({query_embedding.shape[1]}) does not match index dimension ({dimension}).")
            continue

        # Querying the database, using the query, to retrieve slides
        try:
            distances, indices = index.search(query_embedding.astype('float32'), min(num_of_slides * 2, len(embeddings)))
        except Exception as e:
            print(f"Error during search: {str(e)}")
            continue

        # Filtering the results based on relevance of different elements
        results = []
        for i, (distance, idx) in enumerate(zip(distances[0], indices[0])):
            document = all_data['documents'][idx]
            metadata = all_data['metadatas'][idx]
            ids = all_data['ids'][idx]
            text_content, table_content, images_summary_content = separate_content(document)
            text_relevance_score = get_relevance_score(query_text, text_content)
            table_relevance_score = get_relevance_score(query_text, table_content)
            images_summary_relevance_score = get_relevance_score(query_text, images_summary_content)

            # Calculating the combined score, giving different weights to different sections
            relevance_score = 0.6*(text_relevance_score) + 0.2*(table_relevance_score) + 0.2*(images_summary_relevance_score)

            similarity_score = 1 / (1 + distance)
            similarity_score = (similarity_score - 0.5)*2

            alpha = 0.3
            combined_score = alpha * relevance_score + (1 - alpha) * similarity_score

            # Storing the results
            results.append({
                'ids': ids,
                'document': document,
                'metadata': metadata,
                'distance': distance,
                'length': len(text_content),
                'text_relevance_score': text_relevance_score,
                'table_relevance_score': table_relevance_score,
                'images_summary_relevance_score': images_summary_relevance_score,
                'similarity_score': similarity_score,
                'relevance_score': relevance_score,
                'combined_score': combined_score
            })

        # Filtering out the slides that are too similar
        results = [r for r in results if r['ids'] not in excluded_files]
        results.sort(key=lambda x: x['combined_score'], reverse=True)
        image_paths = {f"{ppt_data_path}/{r['metadata']['filename']}/{r['metadata']['slide_number']}.png" : r['ids'] for r in results}
        unique_slides = filter_unique_images(image_paths, used_hashes)
        results = [r for r in results if r['ids'] in unique_slides.values()]

        # There are two modes - get the n best retrieved slides, or get the n most dissimilar retrieved slides
        # Uncomment the one that you want to use, and comment the other one.

        #best_result = find_farthest_slides(results,num_of_slides)
        best_result = results[:num_of_slides]

        best_result = sorted(best_result, key=lambda x: x['combined_score'], reverse=True)

        # Printing the results
        for i in range(len(best_result)):
          excluded_files.add(best_result[i]['ids'])
          filename = best_result[i]['metadata']['filename']
          slide_number = best_result[i]['metadata']['slide_number']
          print(f"\nResult {i+1} for query: '{query_text}'")
          print(f"Filename: {best_result[i]['metadata']['filename']}")
          print(f"Slide Number: {best_result[i]['metadata']['slide_number']}")
          print(f"Distance: {best_result[i]['distance']:.4f}")
          print(f"Length : {best_result[i]['length']}")
          print(f"Text Relevance Score: {best_result[i]['text_relevance_score']:.4f}")
          print(f"Table Relevance Score: {best_result[i]['table_relevance_score']:.4f}")
          print(f"Images Summary Relevance Score: {best_result[i]['images_summary_relevance_score']:.4f}")
          print(f"Relevance Score: {best_result[i]['relevance_score']:.4f}")
          print(f"Combined Score: {best_result[i]['combined_score']:.4f}")
          print(f"Content: {best_result[i]['document'][:100]}...")  # Print first 100 characters
          print("-" * 50)
          all_results.append(best_result[i])

    return {
        'documents': [r['document'] for r in all_results],
        'metadatas': [r['metadata'] for r in all_results],
        'distances': [r['distance'] for r in all_results],
        'text_relevance_scores': [r['text_relevance_score'] for r in all_results],
        'table_relevance_scores': [r['table_relevance_score'] for r in all_results],
        'images_summary_relevance_scores': [r['images_summary_relevance_score'] for r in all_results],
        'relevance_scores': [r['relevance_score'] for r in all_results],
        'combined_scores': [r['combined_score'] for r in all_results]
    }

# Pass the outlines here


In [ ]:
queries = [ {"Introduction to AlgoAnalytics - About us, our team, management, founders, offerings and expertise" : 3},
            {"GenAI in manufacturing  -  different use cases, the products developed for manufacturing sector using AI" : 4},
            {"Computer Vision - different technologies used, various products using CV" : 5},
            {"Large Language Models -  about LLMs, various use cases, products developed that use LLMs" : 4}]

faiss_results = query_chroma(queries,use_hnsw=True, normalize=True)


Result 1 for query: 'Introduction to AlgoAnalytics - About us, our team, management, founders, offerings and expertise'
Filename: 5. AlgoAnalytics_Healthcare and Pharma_overview
Slide Number: slide_2
Distance: 0.2490
Length : 820
Text Relevance Score: 0.9536
Table Relevance Score: 0.0002
Images Summary Relevance Score: 0.0002
Relevance Score: 0.5723
Combined Score: 0.5926
Content: Text:
AlgoAnalytics - An Overview
Who we are
We are a specialist Data Analytics company focused on b...
--------------------------------------------------

Result 2 for query: 'Introduction to AlgoAnalytics - About us, our team, management, founders, offerings and expertise'
Filename: 10. Manufacturing capabilities_AlgoAnalytics_June 24
Slide Number: slide_5
Distance: 0.3004
Length : 659
Text Relevance Score: 0.8570
Table Relevance Score: 0.0000
Images Summary Relevance Score: 0.0002
Relevance Score: 0.5142
Combined Score: 0.5309
Content: Text:
AlgoAnalytics - Offerings and Gen Al Expertise
Core Offerings
Ge

# Print the results in a tabular format

In [ ]:
result_index = -1
for query in queries:
  for query,num in query.items():
      print(f"\nResults for query: '{query}'")
      print("-" * 150)
      print("{:<70} {:<15} {:<15} {:<20} {:<15}".format("Filename", "Slide No.", "Distance", "Relevance Score", "Combined Score"))
      print("-" * 150)
      for _ in range(num):  # Print the results for each query
          result_index += 1
          if result_index < len(faiss_results['metadatas']):
              filename = faiss_results['metadatas'][result_index]['filename']
              slide_no = faiss_results['metadatas'][result_index]['slide_number']
              distance = faiss_results['distances'][result_index]
              relevance_score = faiss_results['relevance_scores'][result_index]
              combined_score = faiss_results['combined_scores'][result_index]
              print("{:<70} {:<15} {:<15.4f} {:<20.4f} {:<15.4f}".format(filename, slide_no, distance, relevance_score,combined_score))
      print("-" * 150)



Results for query: 'Introduction to AlgoAnalytics - About us, our team, management, founders, offerings and expertise'
------------------------------------------------------------------------------------------------------------------------------------------------------
Filename                                                               Slide No.       Distance        Relevance Score      Combined Score 
------------------------------------------------------------------------------------------------------------------------------------------------------
5. AlgoAnalytics_Healthcare and Pharma_overview                        slide_2         0.2490          0.5723               0.5926         
10. Manufacturing capabilities_AlgoAnalytics_June 24                   slide_5         0.3004          0.5142               0.5309         
10. Manufacturing capabilities_AlgoAnalytics_June 24                   slide_22        0.3177          0.3319               0.4620         
------------------

# Printing the results in a detailed format

In [ ]:
for i,(distance,metadata,document) in enumerate(zip(faiss_results['distances'],faiss_results['metadatas'],faiss_results['documents'])):
    print(f"Query_Score_{i} : {1 - distance}\n")
    print(f"Length : {len(document)}")
    print(f"Document : {document}")
    print(f"Metadata : {metadata}\n")


Query_Score_0 : 0.7510474920272827

Length : 846
Document : Text:
AlgoAnalytics - An Overview
Who we are
We are a specialist Data Analytics company focused on building Al/ML powered solutions to solve business problems
* We are an NTT Data invested company *
Services
What makes us Special
100+ Al solution implementation experience
Work across domains, business problems and multi-model data
Strong capabilities in Classical ML, Deep learning as well as emerging tech like Quantum Computing and Digital Twin
Emerging Technologies
Client locations
Our Partners intel)
NTTDaTa
Microsoft
aws
Products
Strictly Confidential
Al-powered video surveillance
Our Offerings
Custom AI Solution Development
AI R&D and Consulting
Cloud Data Engineering
Large Language/ Foundational Models
Quantum Computing
Digital Twin
:FAlgoFabric
aQuality AI-powered parts inspection
NLP based analytics platform

Images Summary:
[]
Metadata : {'filename': '5. AlgoAnalytics_Healthcare and Pharma_overview', 'slide_number': 's

# Creating a PDF of the final retrieved slides


In [ ]:
import fitz
import re
import os

slides = faiss_results['metadatas']

pdf_pages = []

for item in slides:
    ppt_name = item['filename'].replace('.pptx', '')
    slide_number = int(item['slide_number'].replace('slide_',''))

    # Construct the full file path for the image
    pdf_path = f"{ppt_data_path}/{ppt_name}/{ppt_name}.pdf"

    if os.path.exists(pdf_path):
        pdf_pages.append((pdf_path, slide_number - 1))
    else:
        print(f"PDF file not found: {pdf_path}")

# Create a new PDF document
output_pdf = fitz.open()

for pdf_path, page_num in pdf_pages:
    try:
        with fitz.open(pdf_path) as src_pdf:
            if 0 <= page_num < len(src_pdf)+1:
                output_pdf.insert_pdf(src_pdf, from_page=page_num, to_page=page_num)
                print(f"Page {page_num + 1} inserted")
            else:
                print(f"Page {page_num + 1} not found in {pdf_path}")
    except Exception as e:
        print(f"Error processing {pdf_path}: {str(e)}")

# Save the PDF
output_pdf.save(f"{output_pdf_path}/retrieved_slides.pdf")
output_pdf.close()

print('PDF with combined slides created successfully!')

Page 2 inserted
Page 5 inserted
Page 3 inserted
Page 13 inserted
Page 17 inserted
Page 7 inserted
Page 19 inserted
Page 16 inserted
Page 7 inserted
Page 2 inserted
Page 3 inserted
Page 9 inserted
PDF with combined slides created successfully!


# Converting the PDF to PPT

In [ ]:
import convertapi
convertapi.api_secret = '4EWIKNNnMNmQzQZ2'
convertapi.convert('pptx', {
    'File': f"{output_pdf_path}/retrieved_slides.pdf"
}, from_format = 'pdf').save_files(output_ppt_path)